# WS 4 - Regression and Uncertainty Quantification

**Machine Learning in Computational Physics** · University of Vienna

---


<div class="alert alert-block alert-danger">

**Assignment submission**

When you submit this notebook for assignment, make sure that all tasks below (blue boxes) are completed (withc ode) and all questions are answered (with extra markdown responses). Make sure that the notebook runs correctly when all cells are executed in the right order from top to bottom. Before submission, do not clear outputs. Leave all the outputs from the last run included.
</div>

## 1. Motivation and Data Preparation

In this workshop, we look at regression models, hyperparameter optimisation and uncertainty quantification.

After this workshop, you should be able to 

- train and test linear and kernel-based regression models for properties of molecules and materials
- perform hyperparameter optimisation to find optimal model parameters
- use uncertainty quantification approaches to validate and calibrate model predictions

We will again use the data and representations from WS2

In [ ]:
#basic stuff
import os
import sys
from functools import partial
import numpy as np
import random

import ase
from ase.io import read, write
from ase.visualize import view
from ase.build import molecule
from ase.db import connect

from matplotlib import pyplot as plt
import matplotlib as mpl
import pandas
import seaborn as sns
from weas_widget import WeasWidget

#ML stuff
import dscribe
import sklearn
from sklearn.metrics.pairwise import pairwise_kernels
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm # progress bars for loops



from pathlib import Path

# Locate repo root (contains pyproject.toml) — works in Docker, VS Code, and local Jupyter
_p = Path().resolve()
while not (_p / 'pyproject.toml').exists() and _p != _p.parent:
    _p = _p.parent
DATA_DIR = _p / 'data'
%matplotlib inline

Preparing cyclohexane data

In [ ]:
# read in the frames from each MD simulation
traj = []
names = ['chair', 'twist-boat', 'boat', 'half-chair', 'planar']
rgb_colors = [(0.13333333333333333, 0.47058823529411764, 0.7098039215686275),
                (0.4588235294117647, 0.7568627450980392, 0.34901960784313724),
                (0.803921568627451, 0.6078431372549019, 0.16862745098039217),
                (0.803921568627451, 0.13725490196078433, 0.15294117647058825),
                (0.4392156862745098, 0.2784313725490196, 0.611764705882353),]

ranges = np.zeros((len(names), 2), dtype=int)
conf_idx = np.zeros(len(names), dtype=int)

for i, n in enumerate(names):
    # here comes the parsing of the files
    frames = read(DATA_DIR / 'cyclohexane_data' / 'MD' / f'{n}.xyz', '::') #creates a list of Atoms objects.

    for frame in frames:
        # wrap each frame in its box, this means that we ensure that the coordinates of each atom are between 0 and the box size, which is important for the featurization step later on.
        frame.wrap(eps=1E-10)

        # mask each frame so that descriptors are only centered on carbon (#6) atoms
        mask = np.zeros(len(frame))
        mask[np.where(frame.numbers == 6)[0]] = 1 #the mask is list of 0s and 1s, where 1 indicates that the atom is a carbon atom and 0 indicates that it is not. This is used to tell the featurizer which atoms to center the descriptors on.
        frame.arrays['center_atoms_mask'] = mask

    #now we create three lists that contain the whole dataset. 
    
    # ranges is a list of tuples that indicate the start and end index of each trajectory in the traj list and traj is a list of all frames from all trajectories.
    ranges[i] = (len(traj), len(traj) + len(frames)) #list of data ranges

    conf_idx[i] = len(traj) # handy list to indicate the index of the first frame for each trajectory

    traj = [*traj, *frames] # full list of frames, concatenated to a list with 50000 entries

In [ ]:
# energies of the simulation frames
energy = np.array([a.info['energy_eV'] for a in traj])

# energies of the known conformers
c_energy = np.array([traj[c].info['energy_eV'] for c in conf_idx])

# extrema for the energies
max_e = max(energy)
min_e = min(energy)

print('energy range goes from {0:10.6f} to {1:10.6f} eV'.format(min_e, max_e))


Preparing QM7 data

In [ ]:
def get_qm7_energies(filename, key="dft"):
    """ Returns a dictionary with heats of formation for each xyz-file.
    """

    f = open(filename, "r")
    lines = f.readlines()
    f.close()

    energies = []

    for line in lines:
        tokens = line.split()

        xyz_name = tokens[0]
        hof = float(tokens[1])
        dftb = float(tokens[2])

        if key=="delta":
            energies.append(hof - dftb)
        else:
            energies.append(hof)

    return energies

# Import QM7, already parsed to QML
from ase.io import read

qm7_dft_energy = get_qm7_energies(DATA_DIR / 'qm7' / 'hof_qm7.txt', key="dft")
qm7_delta_energy = get_qm7_energies(DATA_DIR / 'qm7' / 'hof_qm7.txt', key="delta")

qm7 = read(DATA_DIR / 'qm7' / 'qm7.xyz', '::') #creates a list of Atoms objects.


## 2. Feature Generation

Let's generate a range of features as we did in WS2. We will use global molecular features

### MBTR

### SOAP

In [ ]:
import collections.abc
collections.Iterable = collections.abc.Iterable # this is unfortunately necessary for compatbility of describe with Py>=3.10
from dscribe.descriptors import SOAP

species = ["H", "C"]
r_cut = 3.5 #cutoff in Angstrom
n_max = 4 # 
sigma = 0.3 #stdev of gaussians
l_max = 4

# Setting up the SOAP descriptor
soap = SOAP(
    species=species,
    periodic=False,
    r_cut=r_cut,
    n_max=n_max,
    l_max=l_max,
    sigma=sigma,
    average='inner', #this extra keyword makes it a global descriptor by averaging over all atoms in the molecule
)

soaps_low = soap.create(traj) #calculate SOAP for all frames

species = ["H", "C"]
r_cut = 5 #cutoff in Angstrom
n_max = 6 # 
sigma = 0.3 #stdev of gaussians
l_max = 6

# Setting up the SOAP descriptor
soap = SOAP(
    species=species,
    periodic=False,
    r_cut=r_cut,
    n_max=n_max,
    l_max=l_max,
    sigma=sigma,
    average='inner', #this extra keyword makes it a global descriptor by averaging over all atoms in the molecule
)

soaps_high = soap.create(traj) #calculate SOAP for all frames


Now we have generated three sets of descriptors

- MBTRs with 2 body interactions (distances only)
- SOAPs with 3 body interactions (low cutoff and angular momentum)
- SOAPS with 3 body interactions (higher cutoff and angular momentum)

In [ ]:
import collections.abc
collections.Iterable = collections.abc.Iterable # this is unfortunately necessary for compatbility of dscribe with Py>=3.10
from dscribe.descriptors import MBTR

min_dist = 0.8  #smallest distance in Angstrom
max_dist = 6.0  #largest distance in Angstrom
#we are constructing MBTR with inverse distances
grid_min = 1.0/max_dist # 1/Angstrom
grid_max = 1.0/min_dist # 1/Angstrom
grid_n = 100 # number of grid points
sigma = 0.05

#k=1, 1-body
#"atomic_number": The atomic number.
#k=2, 2-body
#"distance": Pairwise distance in angstroms.
#"inverse_distance": Pairwise inverse distance in 1/angstrom.
#k=3, 3-body
#"angle": Angle in degrees.
#"cosine": Cosine of the angle.

# MBTR setup for cyclohexane dataset
mbtr = MBTR(
    species=["H", "C"],
    geometry={"function": "inverse_distance"}, # "replace inverse_distance" with "cosine" to get 3-body terms instead of 2-body terms"
    grid={"min": grid_min, "max": grid_max, "n": grid_n, "sigma": sigma},
    weighting={"function": "exp", "scale": 0.5, "threshold": 1e-3},
    periodic=False,
    normalization="l2",
    sparse=False
)

mbtrs = mbtr.create(traj) #calculate MBTR for all frames


## 3. Data Preparation

In a later tutorial, you will learn all about neural networks and deep learning models for interatomic potentials. Here, we will look at models based on linear regression and Kernel Ridge Regression as they form a good starting point. Our aim is to create a model that can predict the energy of any cyclohexane conformation. The previously calculated feature vectors (`descriptors`) are our input quantities ($\mathbf{x}$), the energies are our labels.




In [ ]:
#pick either 
descriptors = soaps_low
#descriptors = soaps_high
#descriptors = mbtrs

First we have to split our data into train and test data. The train data is what we will use to train the model. The test data will be "unseen" and used to evaluate the prediction capabilities of the model.

We won't be training on the full training data set, but only on a small subset. We will take 1000 data points.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

#we know the `energy`` of each frame, the energy of the starting conformers (c_energy), max_e, and min_e

X = descriptors
#our labels are the energies
y = energy

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, #inputs and outputs
    test_size=0.20, #20% of the data is test data
    random_state=42, #a random seed
    ) 

#only train on a small subset of samples rather than the full training dataset
X_train, y_train = resample(X_train_full, y_train_full, replace=False, n_samples=1000, random_state=42) #note how we fix the random seed to make this reproducible


Next, we standardize the training and test sets. This means scaling and normalizing the data. This way, the model mean is centred around zero and the model does not have to cover large value ranges. We do this with the inputs and the outputs.

For the labels/outputs, this is as simple as
$$
\tilde{\mathbf{y}} = \frac{\mathbf{y} - u}{s}
$$
where u and s are the mean and the standard deviation of the labels.

In [ ]:
from sklearn import preprocessing

#scikit-learn has a utility to help with the feature vectors

scaler = preprocessing.StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test) #? This needs to be the same scaler as for the training data, otherwise the test data would be scaled into the wrong space

#rescaling the labels

u = np.mean(y_train)
s = np.std(y_train)

y_train_scaled = (y_train-u)/s
y_test_scaled = (y_test-u)/s

def unscale(y):
    # this is for transforming the predictions back into proper energies
    return (y*s)+u


**Task for you**
<div class="alert alert-block alert-info">
    
- Plot a histogram of energies before and after the shift-and-scale operation
- The cyclohexane dataset provides the energy of each conformer along the trajectories (`energy`), but also the energy for each of the five starting conformer (`c_energy`). If you apply the shift-and-scale operation to the reference energies, where do they lie in the energy histogram? Can they be easily distinguished?

Consider the energy range in the data and the energy difference between conformations: 
- Performance test: Your model needs to be able to distinguish the five starting geometries by their energy within a 95% confidence interval? What is the accuracy threshold (e.g. in MAE) and the variance that is required? 
- Write a function that takes a model (or list of models) and predicts the energy of the five starting geometries and the associated variance. The function should calculate whether the energies are significant within a 99%CI. You can use dummy prediction values to test the function. We will use this function at the end to assess the models.

</div>

## 4. Multiple Linear Regression

In multiple linear regression, we try to find a set of regression coefficients $\mathbf{w}$ to approximate a $N$-dimensional function

$$
f(\mathbf{x};\mathbf{\beta}) = \sum_{j=1}^{N} w_j x_j = \mathbf{x}^T \mathbf{w}
$$

In our case, we would like to predict the energy for an arbitrary conformation of cyclohexane. We have already acknowledged that this would not be possible based simply on the Cartesian positions of atoms, which is why we have created various representations that provide us with high dimensional feature vectors (e.g. SOAP).

The feature vectors form the inputs $\mathbf{x}$ and the dimensionality of the space ($N$) in which we want to fit a linear relationship is the length of the descriptor. The quality of the fit will depend on the ability of the high dimensional space spanned by the descriptor to express complex relationships as linear trends. The more chemically meaningful (and higher dimensional) the descriptor, the better the fit. 

Let's test this for the energy prediction.


In [ ]:
from sklearn.linear_model import LinearRegression

multiple_linear_regression = LinearRegression()
multiple_linear_regression.fit(X = X_train, y = y_train_scaled)
# This is the training (or fitting) part

Determine $R^2$, MAE, and RMSD of the fit

$R^2$ metric (Coefficient of determination)
$$
R^2 = 1 - \frac{\sum_i(y_i-f_i)^2}{\sum_i(y_i-u)^2}
$$
where $u$ is the mean of the observed data, $y_i$ is the data value ("ground truth") and $f_i$ is the prediction.

Mean Absolute Error (MAE)
$$
\mathrm{MAE} = \frac{\sum_i |y_i - f_i|}{n}
$$

Root Mean Square Error (RMSE)
$$
\mathrm{RMSE} = \sqrt{\frac{\sum_{i} (y_i - f_i)^2}{n} }
$$

In [ ]:
# This is the prediction part
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

pred_train_lr = unscale(multiple_linear_regression.predict(X_train)) #predict and unscale the predictions for the training data
pred_test_lr = unscale(multiple_linear_regression.predict(X_test)) #predict and unscale the predictions for the test data

r2 = r2_score(y_test, pred_test_lr)
print('R2 train = ', r2_score(y_train, pred_train_lr))
print('R2 test = ', r2_score(y_test, pred_test_lr))

mae = mean_absolute_error(y_test, pred_test_lr)
print('MAE test = ', mae)
rmse = np.sqrt(mean_squared_error(y_test,pred_test_lr ))
print('RMSE test = ', rmse)

In [ ]:
# Plotting the predictions vs the real values for both the training and test set

plt.scatter(y_test,pred_test_lr, label='Test Set')
plt.scatter(y_train,pred_train_lr, label='Training Set', alpha=0.5)

plt.xlabel('Real')
plt.ylabel('Predicted')
plt.legend()

**Task for you**

<div class="alert alert-block alert-info">
    
- Which of the descriptors that we have prepared gives the best correlation and test error with a simple linear fit? Plot the results alongside each other.
- Do any of them already achieve the required accuracy?
</div>

## 5. Kernel Ridge Regression (KRR) and Gaussian Process Regression (GPR)

### 5.1 Kernel Ridge Regression

**Kernel Ridge Regression** provides a rigorous mathematical framework to extend linear statistical learning techniques to model nonlinear dependencies in high-dimensional data. The method is based on measuring similarity between pairs of data points and then using this similarity to unfold nonlinearities in the original data space. 
$$
f(\phi(\mathbf{x}')) = \sum_{p=1}^{P} \alpha_p k(\mathbf{x}_p,\mathbf{x}')
$$
where $k(\mathbf{x}_p,\mathbf{x}')$ is the dot product in feature space (the so-called kernel). The kernel is the similarity measure with which we compare training datapoint $\mathbf{x}_p$ and the point to be predicted $\mathbf{x}'$.

We can choose a linear kernel were $k(\mathbf{x}_p,\mathbf{x}') = \langle \mathbf{x}_p|\mathbf{x}' \rangle $ (simple dot product) or we can choose a Gaussian kernel:
$$ 
k(\mathbf{x}_p,\mathbf{x}') = \exp \left(-\frac{1}{2\sigma^2}\sum_j^{B} (x_{ij} - x_j^{'})^2 \right) 
$$
This is equivalent to mapping the input vector to a space spanned by Gaussian functions (with width $\sigma$) centred at the training data points.

The term **Ridge Regression** comes from the fact that a simple inversion of the equation $\mathbf{y}=\mathbf{K}\mathbf{\alpha}$ to determine $\mathbf{\alpha}$ is numerically unstable. This is often connected with individual parameters becoming extremely large. In Ridge regression, we use a regularizer, where we minimize the L2 loss between prediction and training data under the constraint that the coefficients remain small (Tikhonov regularization). That means, instead of solving
$$ \mathbf{\alpha} = \mathbf{K}^{-1}\mathbf{y} $$
we solve
$$ 
\mathbf{\alpha} = (\mathbf{K} + \lambda\mathbf{I})^{-1}\mathbf{y}
$$
where $\mathbf{I}$ is the identity matrix and $\lambda$ is a small nonnegative smoothing parameter (regularization parameter). Regularization is crucial to reduce the risk of overfitting.

Gaussian kernel fits are defined by two hyperparameters, $\gamma = 1/2\sigma^2$ and $\lambda$. If $\gamma$ is too large (Gaussians too narrow), the model memorises training points but fails to generalise; if $\gamma$ is too small (Gaussians too wide), the model is too smooth to capture detail.

<div class="alert alert-block alert-danger">

Typically, hyperparameters within the ML model need to be optimised to achieve optimal accuracy and generalisability.
</div>

Gaussian Process Regression (Section 5.2) is derived from the same kernel framework but treats the problem probabilistically, giving uncertainty estimates as a natural byproduct.

Let's start by reproducing the linear fit with a linear kernel

In [ ]:
from sklearn.kernel_ridge import KernelRidge

krr = KernelRidge(alpha=0.00001, kernel='linear') # alpha here is lambda in the explanation above
krr.fit(X_train, y_train_scaled)


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

pred_train_kr =  unscale(krr.predict(X_train))
pred_test_kr = unscale(krr.predict(X_test))

r2 = r2_score(y_test, pred_test_kr)
print('R2 train = ', r2_score(y_train, pred_train_kr))
print('R2 test = ', r2_score(y_test, pred_test_kr))

mae = mean_absolute_error(y_test, pred_test_kr)
print('MAE test = ', mae)
rmse = np.sqrt(mean_squared_error(y_test,pred_test_kr ))
print('RSME test = ', rmse)

plt.scatter(y_test,pred_test_kr, label='Test Set', alpha=0.3)
plt.scatter(y_train,pred_train_kr, label='Training Set')


plt.xlabel('Real')
plt.ylabel('Predicted')
plt.legend()

Next we try the Gaussian (RBF) kernel. Here we have to set the width of the RBFs

In [ ]:
from sklearn.kernel_ridge import KernelRidge

krr = KernelRidge(alpha=0.00001, kernel='rbf', gamma=0.005)
krr.fit(X_train, y_train_scaled)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

pred_train_kr =  unscale(krr.predict(X_train))
pred_test_kr = unscale(krr.predict(X_test))

r2 = r2_score(y_test, pred_test_kr)
print('R2 train = ', r2_score(y_train, pred_train_kr))
print('R2 test = ', r2_score(y_test, pred_test_kr))

mae = mean_absolute_error(y_test, pred_test_kr)
print('MAE test = ', mae)
rmse = np.sqrt(mean_squared_error(y_test,pred_test_kr ))
print('RSME test = ', rmse)

plt.scatter(y_test,pred_test_kr, label='Test Set', alpha=0.3)
plt.scatter(y_train,pred_train_kr, label='Training Set')

plt.xlabel('Real')
plt.ylabel('Predicted')
plt.legend()

### 5.2 Gaussian Process Regression

**Gaussian Process Regression (GPR)** shares the same predictive form as KRR — a kernel-weighted sum over training points — but is derived from a Bayesian framework. We place a Gaussian process prior over functions and condition it on the training data. The posterior is also Gaussian and delivers both a **mean prediction** and a **predictive variance** for every new input, with no extra steps.

The predictive standard deviation is:
$$
\sigma(\mathbf{x}') = \sqrt{k(\mathbf{x}',\mathbf{x}') - \mathbf{k}^T(\mathbf{K} + \lambda\mathbf{I})^{-1}\mathbf{k}}
$$
where $\mathbf{k}$ is the vector of kernel values between the new point and all training points. The variance is large in data-sparse regions (uncertain) and small where training data is dense (confident). GPR can also tune its own kernel hyperparameters by maximising the **log-marginal likelihood** of the training data.

**Practical limitation**: GPR requires inverting an $N \times N$ kernel matrix — $\mathcal{O}(N^3)$ scaling. We use 300 training points here to keep the computation fast.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.utils import resample as sk_resample

# GPR is O(N^3) in training set size — use 300 points to keep it tractable
X_gpr_raw, y_train_gpr = sk_resample(X_train_full, y_train_full,
                                      replace=False, n_samples=300, random_state=42)
# Scale using the same scaler that was fit on the 1000-pt training set (consistent with X_test)
X_train_gpr = scaler.transform(X_gpr_raw)
X_test_gpr  = X_test  # already scaled

# GPR is sensitive to the scale of the labels, so we standardize them here. Note that we use the mean and std of the GPR training set, not the 1000-pt training set, to be consistent with the inputs to GPR.
u_gpr = np.mean(y_train_gpr)
s_gpr = np.std(y_train_gpr)
y_train_gpr_scaled = (y_train_gpr - u_gpr) / s_gpr

# RBF + WhiteKernel (observation noise); hyperparameters optimised via log-marginal likelihood
# We add observation noise to make the optimization landscape smoother and thus easier to optimize. This is a common practice in GPR, especially when the data is noisy, as it can help prevent overfitting and improve the generalization performance of the model.
kernel_gpr = 1.0 * RBF(length_scale=1.0) + WhiteKernel(noise_level=0.01)
# define the GPR model with the kernel and fit it to the training data. The hyperparameters of the kernel (length scale and noise level) will be optimized by maximizing the log-marginal likelihood of the training data.
gpr = GaussianProcessRegressor(kernel=kernel_gpr, n_restarts_optimizer=2, normalize_y=False)
gpr.fit(X_train_gpr, y_train_gpr_scaled)
print("Optimised kernel:", gpr.kernel_)

In [ ]:
# make predictions with GPR on the test set, including the predictive uncertainty (standard deviation)
pred_gpr_scaled, sigma_gpr_scaled = gpr.predict(X_test_gpr, return_std=True)
pred_gpr  = pred_gpr_scaled * s_gpr + u_gpr
sigma_gpr = sigma_gpr_scaled * s_gpr  # rescale uncertainty to eV

print(f'GPR MAE (test)    = {mean_absolute_error(y_test, pred_gpr):.5f} eV')
print(f'Mean predictive σ = {np.mean(sigma_gpr):.5f} eV')

sort_idx    = np.argsort(y_test)[::50] #visualise only every 50th point to avoid overcrowding the plot
y_sorted    = np.array(y_test)[sort_idx]
pred_sorted = pred_gpr[sort_idx]
sig_sorted  = sigma_gpr[sort_idx]


# plotting the GPR predictions vs the real values, including the uncertainty band
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test, pred_gpr, s=5, alpha=0.5)
lims = [min(y_test), max(y_test)]
axes[0].plot(lims, lims, 'k--', lw=1)
axes[0].set_xlabel('Reference energy (eV)')
axes[0].set_ylabel('GPR mean prediction (eV)')
axes[0].set_title('GPR parity plot on test data')

axes[1].plot(y_sorted, color='k', lw=1, label='Reference')
axes[1].plot(pred_sorted, color='C0', lw=1, label='GPR mean')
axes[1].fill_between(np.arange(len(pred_sorted)),
                     pred_sorted - 2*sig_sorted,
                     pred_sorted + 2*sig_sorted,
                     alpha=0.3, label='±2σ')
axes[1].set_xlabel('Test sample (sorted by energy)')
axes[1].set_ylabel('Energy (eV)')
axes[1].set_title('GPR predictions with uncertainty band')
axes[1].legend()
plt.tight_layout()

Note how the model fits the trained data almost perfectly and much better than the linear model, but does not capture the test data quite as well. This means we are overfitting. This can have several reasons:

- The data is simple and the model is too complex
- The feature space is too large
- Too much noise and outliers in training data (can be excluded here)
- Not enough data
- The model is not sufficiently constrained and the hyperparameters are not optimal.

We need to find a way to properly evaluate and fine-tune the model.

**Task for you**

<div class="alert alert-block alert-info">

- Use the variance from the GPR to run the performance test we defined above for the 5 starting energies

</div>

## 6. Cross Validation

Cross validation (CV) is a resampling method that uses different portions of data to test and train models. It is a way to assess how well a model generalises and to robustly evaluate model performance. We can also use cross validation to check for outliers in the dataset. As shown below, the data is split into training and test data. The test data is then further split into $k$ "folds" (here 5). Over $k$ possible permutations, we can change which part of the training data we "hold out". That means we can train $k$ different models on different parts of the data. 
If the standard deviation over several folds is low, the model is robust. If the MAE on the hold-out set is very similar, it likely also has few outliers. The standard deviation can also provide us with a measure of the uncertainty of the model, which could be used in the context of committee-based active learning. 

CV is also used when the space of hyperparameters is explored to find the optimal model layout. For KRR, this is $\sigma$ and $\lambda$, for neural networks, it can be parameters that define the layout of the network. 


![crossval](https://scikit-learn.org/stable/_images/grid_search_cross_validation.png
)

Let's evaluate the mean prediction error and standard deviation for the linear model and the KRR model with given settings.

In [ ]:
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import cross_validate
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer

krr = KernelRidge(alpha=0.00001, kernel='linear')

#this is a dictionary which we can use to pass multiple scoring functions into CV (even custom-made ones)
scoring = {
    'r2_score': make_scorer(r2_score),
    'MAE': make_scorer(mean_absolute_error),
    'RMSE': make_scorer(mean_squared_error),
    }

scores = cross_validate(
    krr, #model
    X_train, #training data
    y_train_scaled, #training data
    scoring=scoring, # types of model scores
    cv=5, # k-foldedness of cross validation
    return_estimator=False, #this function can return all five models if needed
    return_train_score=True
    )

print('Linear Model')
print('Train Errors:')
print('    The mean and stdev of train MAE over the folds are {0:8.5f} and {1:8.5f} eV'.format(np.mean(scores['train_MAE']*s), np.std(np.sqrt(scores['train_MAE'])*s)))
print('    The mean and stdev of train RMSE over the folds are {0:8.5f} and {1:8.5f} eV'.format(np.mean(scores['train_RMSE']*s), np.std(np.sqrt(scores['train_RMSE'])*s)))
print('Test Errors:')
print('    The mean and stdev of test MAE over the folds are {0:8.5f} and {1:8.5f} eV'.format(np.mean(scores['test_MAE']*s), np.std(np.sqrt(scores['test_MAE'])*s)))
print('    The mean and stdev of test RMSE over the folds are {0:8.5f} and {1:8.5f} eV'.format(np.mean(scores['test_RMSE']*s), np.std(np.sqrt(scores['test_RMSE'])*s)))

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer

krr = KernelRidge(alpha=0.00001, kernel='rbf', gamma=0.005)

#this is a dictionary which we can use to pass multiple scoring functions into CV (even custom-made ones)
scoring = {
    'r2_score': make_scorer(r2_score),
    'MAE': make_scorer(mean_absolute_error),
    'RMSE': make_scorer(mean_squared_error),
    }

scores = cross_validate(
    krr, 
    X_train, 
    y_train_scaled, 
    scoring=scoring, 
    cv=5, 
    return_estimator=False, #this function can return all five models if needed
    return_train_score=True
    )

print('RBF Model')
print('Train Errors:')
print('    The mean and stdev of train MAE over the folds are {0:8.5f} and {1:8.5f} eV'.format(np.mean(scores['train_MAE']*s), np.std(np.sqrt(scores['train_MAE'])*s)))
print('    The mean and stdev of train RMSE over the folds are {0:8.5f} and {1:8.5f} eV'.format(np.mean(scores['train_RMSE']*s), np.std(np.sqrt(scores['train_RMSE'])*s)))
print('Validation Errors:')
print('    The mean and stdev of test MAE over the folds are {0:8.5f} and {1:8.5f} eV'.format(np.mean(scores['test_MAE']*s), np.std(np.sqrt(scores['test_MAE'])*s)))
print('    The mean and stdev of test RMSE over the folds are {0:8.5f} and {1:8.5f} eV'.format(np.mean(scores['test_RMSE']*s), np.std(np.sqrt(scores['test_RMSE'])*s)))


**Task for you**

<div class="alert alert-block alert-info">

- What have we learned and what do we have to do next? Which one of the listed causes of overfitting might be the case here?
- What is the better choice for the current problem? Linear kernel or Gaussian kernel?

</div>

## 7. Hyperparameter Optimisation

We can use the CV to perform a systematic hyperparameter optimisation. In the following we use a scikit-learn feature to perform a hyperparameter grid search.

### The Bias-Variance Tradeoff

Any supervised model makes predictions that deviate from the truth for two distinct reasons:

- **Bias** is the systematic error from a model that is too simple to capture the true relationship. A high-bias model *underfits*: it performs poorly on both training and test data.
- **Variance** is the sensitivity of the model to the specific training set drawn. A high-variance model *overfits*: it memorises the training data but fails to generalise to new points.

The total expected squared error on unseen data decomposes as:

$$
\text{Total Error} = \underbrace{\text{Bias}^2}_{\text{systematic, underfitting}} + \underbrace{\text{Variance}}_{\text{random, overfitting}} + \underbrace{\sigma^2_\text{noise}}_{\text{irreducible}}
$$

For KRR with an RBF kernel the two hyperparameters control opposite ends of this tradeoff:

| Hyperparameter | Large value | Small value |
|---|---|---|
| $\gamma$ (RBF width) | narrow Gaussians → more flexible → lower bias, **higher variance** | wide Gaussians → smoother → **higher bias**, lower variance |
| $\lambda$ (regularisation) | strong smoothing → **higher bias**, lower variance | weak smoothing → lower bias, **higher variance** |

The grid search below sweeps $\gamma$ at fixed $\lambda$ and reveals this tradeoff directly in the plot of train vs validation MAE.

In [ ]:
from sklearn.model_selection import cross_validate, GridSearchCV
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer
import pandas as pd

j = 0.000001
alphas = [ j*(10**i) for i in range(8)]
gammas = [ j*(10**(i/8)) for i in range(64)]

# change settings below to vary different hyperparameters and see the effect on the CV scores. Note that this is not an exhaustive search, but just a demonstration of how to use GridSearchCV for hyperparameter optimization. For a more exhaustive search, you could use RandomizedSearchCV or Bayesian optimization methods.

#vary alphas
#parameters = {'kernel':['rbf'], 'alpha':alphas, 'gamma':[0.0001]}
#vary gammas
parameters = {'kernel':['rbf'], 'alpha':[0.001], 'gamma':gammas}

scoring = {
    'r2_score': make_scorer(r2_score),
    'MAE': make_scorer(mean_absolute_error),
    'RMSE': make_scorer(mean_squared_error),
    }

krr = KernelRidge()
gridsearch = GridSearchCV(krr, 
param_grid=parameters, 
scoring=scoring, 
cv=5, 
n_jobs=6,
refit=False, 
return_train_score=True
)

gridsearch.fit(X_train, y_train_scaled)

cvdata = pd.DataFrame(gridsearch.cv_results_)


In [ ]:
#try also printing the full dataframe by uncommenting the next line and commenting the other one out
#cvdata
cvdata[['param_alpha','param_gamma','param_kernel','mean_test_MAE', 'std_test_MAE', 'mean_train_MAE', 'std_train_MAE']]

In [ ]:
cvdata['mean_test_MAE']  = cvdata['mean_test_MAE'].multiply(s)   # unscale to eV
cvdata['mean_train_MAE'] = cvdata['mean_train_MAE'].multiply(s)  # unscale to eV

plotframe = cvdata[['param_gamma', 'mean_test_MAE', 'mean_train_MAE']].rename(
    columns={'mean_test_MAE': 'validation MAE', 'mean_train_MAE': 'train MAE'})

ax = plotframe.plot('param_gamma', logx=True, figsize=(7, 4))
ax.set_xlabel('γ (RBF width parameter)')
ax.set_ylabel('MAE (eV)')
ax.set_title('Bias–variance tradeoff: train vs validation MAE as a function of γ')
ax.legend()
plt.tight_layout()

The optimal $\gamma$ sits at the minimum of the validation curve. The gap between the two curves at that point quantifies how much variance remains; a larger gap means higher overfitting risk.

An alpha value of 0.0001 and a gamma value of 0.001 provide a good balance. Let's use these to reevaluate the model.

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer

krr = KernelRidge(alpha=0.0001, kernel='rbf', gamma=0.001)

#this is a dictionary which we can use to pass multiple scoring functions into CV (even custom-made ones)
scoring = {
    'r2_score': make_scorer(r2_score),
    'MAE': make_scorer(mean_absolute_error),
    'RMSE': make_scorer(mean_squared_error),
    }

scores = cross_validate(
    krr, 
    X_train, 
    y_train_scaled, 
    scoring=scoring, 
    cv=5, 
    return_estimator=False, #this function can return all five models if needed
    return_train_score=True
    )

print('Train Errors:')
print('    The mean and stdev of train MAE over the folds are {0:8.5f} and {1:8.5f} eV'.format(np.mean(scores['train_MAE']*s), np.std(np.sqrt(scores['train_MAE'])*s)))
print('    The mean and stdev of train RMSE over the folds are {0:8.5f} and {1:8.5f} eV'.format(np.mean(scores['train_RMSE']*s), np.std(np.sqrt(scores['train_RMSE'])*s)))
print('Test Errors:')
print('    The mean and stdev of test MAE over the folds are {0:8.5f} and {1:8.5f} eV'.format(np.mean(scores['test_MAE']*s), np.std(np.sqrt(scores['test_MAE'])*s)))
print('    The mean and stdev of test RMSE over the folds are {0:8.5f} and {1:8.5f} eV'.format(np.mean(scores['test_RMSE']*s), np.std(np.sqrt(scores['test_RMSE'])*s)))

**Task for you**

<div class="alert alert-block alert-info">

- Use this grid search code as a starting point to identify the optimal parameters that will provide the lowest test error and will not lead to overfitting of the training data.
- Compare the best possible KRR and the optimised GPR models and their train and test MAEs and stdevs. How do they differ and why?

</div>

## 8. Learning Curves

Maybe we need more training data. Studying how quickly the prediction error goes down as we add training data means assessing the rate of learning of the model. Learning curves are important to understand the ability of models to learn from little data. We can use them to compare across the three descriptors we have generated.


**Task for you**

<div class="alert alert-block alert-info">

- Study the plot below. Which descriptor achieves the lowest error for the smallest training data set?
</div>

In [ ]:
from sklearn import preprocessing
from sklearn.utils import resample
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer
from sklearn.model_selection import cross_validate, train_test_split

descriptors_list = [mbtrs, soaps_low, soaps_high]
ndatas = [100, 500, 1000, 2500, 5500]

learning_curve = np.zeros([len(descriptors_list), len(ndatas), 2])

for i, X_lc in tqdm(enumerate(descriptors_list)):
    X_train_full_lc, X_test_lc, y_train_full_lc, y_test_lc = train_test_split(
        X_lc, y,
        test_size=0.20,
        random_state=42,
    )
    for j, ndata in tqdm(enumerate(ndatas)):

        X_tr_lc, y_tr_lc = resample(X_train_full_lc, y_train_full_lc,
                                     replace=False, n_samples=ndata, random_state=42)
        scaler_lc = preprocessing.StandardScaler()
        scaler_lc.fit(X_tr_lc)
        X_tr_lc_s = scaler_lc.transform(X_tr_lc)

        u_lc = np.mean(y_tr_lc)
        s_lc = np.std(y_tr_lc)
        y_tr_lc_scaled = (y_tr_lc - u_lc) / s_lc

        scoring = {
            'r2_score': make_scorer(r2_score),
            'MAE': make_scorer(mean_absolute_error),
            'RMSE': make_scorer(mean_squared_error),
        }

        krr_lc = KernelRidge(alpha=0.0001, kernel='rbf', gamma=0.001)
        scores_lc = cross_validate(
            krr_lc,
            X_tr_lc_s,
            y_tr_lc_scaled,
            scoring=scoring,
            cv=5,
            n_jobs=6,
            return_estimator=False,
            return_train_score=True,
        )

        learning_curve[i, j, 0] = np.mean(scores_lc['test_MAE']) * s_lc
        learning_curve[i, j, 1] = np.std(scores_lc['test_MAE']) * s_lc

In [ ]:
descriptor_names = ['MBTR', 'SOAP_low', 'SOAP_high']

fig, ax = plt.subplots(figsize=(7, 4))
for i in range(len(descriptors_list)):
    ax.errorbar(ndatas, learning_curve[i, :, 0], learning_curve[i, :, 1],
                label=descriptor_names[i], marker='o', capsize=3)
ax.set_xlabel('Number of training data points')
ax.set_ylabel('MAE (eV)')
ax.set_title('Learning curves')
ax.legend()
plt.tight_layout()

**Task for you**

<div class="alert alert-block alert-info">

- Go back to WS2 and WS3 and take the best-performing SOAP descriptor settings you have identified there. Plot this additional SOAP descriptor as an additional line in the learning curve


</div>

## 9. Uncertainty Quantification

Knowing *how confident* a model is in its predictions is as important as the predictions themselves. In atomistic simulation workflows, uncertainty estimates are used to:

- **Flag extrapolation**: detect when a test structure lies far outside the training distribution
- **Drive active learning**: query only the most uncertain structures for expensive reference calculations, not random ones
- **Guide deployment**: decide when a model is reliable enough to replace the reference method

We will explore two complementary approaches:

1. **Ensemble (committee) uncertainty**: train $N$ models on bootstrap resamples of the training data; the spread of their predictions is the uncertainty estimate.
2. **Calibration**: verify that the uncertainty estimates are statistically meaningful — a well-calibrated model's ±2σ interval should contain ~95% of true values.

For reference, GPR (Section 5) provides a third route: the posterior variance is built directly into the model and requires no additional training.

We first re-establish a clean training set, since the learning curve loop in Section 8 overwrites the scaling variables.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn import preprocessing

# Re-establish a clean training set after the learning curve loop (which overwrites scaling variables)
X_train_full_uq, X_test_uq, y_train_full_uq, y_test_uq = train_test_split(
    soaps_low, y, test_size=0.20, random_state=42)
X_train_uq, y_train_uq = resample(X_train_full_uq, y_train_full_uq,
                                   replace=False, n_samples=1000, random_state=42)
scaler_uq   = preprocessing.StandardScaler().fit(X_train_uq)
X_train_uq_s = scaler_uq.transform(X_train_uq)
X_test_uq_s  = scaler_uq.transform(X_test_uq)

u_uq = np.mean(y_train_uq)
s_uq = np.std(y_train_uq)
y_train_uq_scaled = (y_train_uq - u_uq) / s_uq

def unscale_uq(y_scaled):
    return y_scaled * s_uq + u_uq

### 9.1 Ensemble (Committee) Uncertainty

We train $N$ independent KRR models, each on a different **bootstrap resample** (sampling with replacement) of the training data. For each test point, the committee gives $N$ predictions; their mean is the final prediction and their standard deviation is the epistemic uncertainty estimate:

$$\bar{f}(\mathbf{x}') = \frac{1}{N}\sum_{k=1}^N f_k(\mathbf{x}'), \qquad \sigma(\mathbf{x}') = \sqrt{\frac{1}{N-1}\sum_{k=1}^N \bigl(f_k(\mathbf{x}') - \bar{f}(\mathbf{x}')\bigr)^2}$$

The uncertainty is large when committee members disagree — this occurs in data-sparse regions or when a test point lies outside the training distribution (extrapolation).

In [ ]:
from sklearn.kernel_ridge import KernelRidge
from sklearn.utils import resample

N_committee = 20
committee_preds = np.zeros((N_committee, len(y_test_uq)))

for k in range(N_committee):
    X_boot, y_boot = resample(X_train_uq_s, y_train_uq_scaled, replace=True, random_state=k)
    krr_k = KernelRidge(alpha=0.0001, kernel='rbf', gamma=0.001)
    krr_k.fit(X_boot, y_boot)
    committee_preds[k] = unscale_uq(krr_k.predict(X_test_uq_s))

ensemble_mean = committee_preds.mean(axis=0)
ensemble_std  = committee_preds.std(axis=0)

from sklearn.metrics import mean_absolute_error
print(f'Ensemble MAE  = {mean_absolute_error(y_test_uq, ensemble_mean):.5f} eV')
print(f'Mean σ        = {np.mean(ensemble_std):.5f} eV')

In [ ]:
sort_idx_uq  = np.argsort(y_test_uq)[::50]
y_sorted_uq  = np.array(y_test_uq)[sort_idx_uq]
mean_sorted  = ensemble_mean[sort_idx_uq]
std_sorted   = ensemble_std[sort_idx_uq]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(y_test_uq, ensemble_mean, s=5, alpha=0.4)
lims_uq = [min(y_test_uq), max(y_test_uq)]
axes[0].plot(lims_uq, lims_uq, 'k--', lw=1)
axes[0].set_xlabel('Reference energy (eV)')
axes[0].set_ylabel('Ensemble mean (eV)')
axes[0].set_title('Ensemble parity plot')

axes[1].plot(y_sorted_uq, color='k', lw=1, label='Reference')
axes[1].plot(mean_sorted, color='C0', lw=1, label='Ensemble mean')
axes[1].fill_between(np.arange(len(mean_sorted)),
                     mean_sorted - 2*std_sorted,
                     mean_sorted + 2*std_sorted,
                     alpha=0.3, label='±2σ')
axes[1].set_xlabel('Test sample (sorted by energy)')
axes[1].set_ylabel('Energy (eV)')
axes[1].set_title('Ensemble predictions with uncertainty band')
axes[1].legend()
plt.tight_layout()

**Task for you**

<div class="alert alert-block alert-info">

- Use the variance from the Ensemble to run the performance test we defined above for the 5 starting energies

</div>

### 9.2 Calibration

A model is **well-calibrated** if its stated confidence levels match observed frequencies. If the model claims "90% of predictions lie within ±1.65σ", then roughly 90% of test points should indeed fall in that interval.

We check this with a **calibration curve**: for a sweep of confidence levels $p$, compute the fraction of test points that fall inside the $z_p \cdot \sigma(\mathbf{x}')$ interval (where $z_p$ is the two-sided $p$-quantile of the standard normal) and plot it against $p$.

| Position on the plot | Interpretation |
|---|---|
| On the diagonal | Perfectly calibrated |
| **Above** the diagonal | **Underconfident** — intervals too wide; model overly cautious |
| **Below** the diagonal | **Overconfident** — intervals too narrow; actual errors larger than claimed |


In [ ]:
from scipy import stats

residuals_ens = np.array(y_test_uq) - ensemble_mean
z_scores_ens  = residuals_ens / (ensemble_std + 1e-10)

confidence_levels = np.linspace(0.05, 0.99, 50)
observed_coverage = [np.mean(np.abs(z_scores_ens) <= stats.norm.ppf((1 + p) / 2))
                     for p in confidence_levels]

plt.figure(figsize=(5, 5))
plt.plot(confidence_levels, observed_coverage, marker='o', ms=3, label='Ensemble model')
plt.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
plt.xlabel('Expected coverage')
plt.ylabel('Observed coverage')
plt.title('Calibration curve')
plt.legend()
plt.tight_layout()

mean_cal_err = np.mean(np.abs(np.array(observed_coverage) - confidence_levels))
print(f'Mean calibration error: {mean_cal_err:.4f}')

**Task for you**

<div class="alert alert-block alert-info">

- Is the ensemble model overconfident or underconfident based on the calibration curve? What does this mean for how you would use these uncertainty estimates in practice?
- Can you show that the ensemble variance $\sigma$ correlates with the actual prediction error $|y - \bar{f}|$? Make a scatter plot of $\sigma$ vs $|y - \bar{f}|$ for the test set. What would a *useful* uncertainty estimate look like, and how does this model compare?
</div>

## 10. Delta Learning

In many scientific applications we have a **cheap, approximate** computational method and a **costly, accurate** reference. Rather than training an ML model to predict the expensive target directly, we train it on the *difference* — the **delta** (or correction):

$$\Delta E = E_{\text{accurate}} - E_{\text{cheap}}$$

The final prediction is then:

$$E_{\text{accurate}}(\mathbf{x}') \approx E_{\text{cheap}}(\mathbf{x}') + f_\Delta(\mathbf{x}')$$

**Why does this work?** The cheap method (here DFTB3 tight-binding) already captures most of the physics — bonding, molecular size, composition. What remains is a smooth, systematic correction between the two theories. That correction is a simpler function to learn than the total energy, and it is smaller in magnitude, so the model needs less capacity and less data to reach the same accuracy.

This approach — called **Δ-ML** — was introduced by Ramakrishnan et al. (*J. Chem. Theory Comput.* 2015) and is now common practice in ab initio machine learning.

The QM7 dataset provides exactly this setting: `qm7_dft_energy` contains PBE0 DFT heats of formation, and `qm7_delta_energy` contains the PBE0 − DFTB3 correction, both loaded in Section 1. The histograms below show how much narrower the delta distribution is.

In [ ]:
y_direct_arr = np.array(qm7_dft_energy)
y_delta_arr  = np.array(qm7_delta_energy)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(y_direct_arr, bins=60, color='C0', edgecolor='none', density=True)
axes[0].set_xlabel('PBE0 heat of formation (kcal/mol)')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Direct target\nrange = {y_direct_arr.max()-y_direct_arr.min():.0f} kcal/mol')

axes[1].hist(y_delta_arr, bins=60, color='C1', edgecolor='none', density=True)
axes[1].set_xlabel('Δ (PBE0 − DFTB3)  (kcal/mol)')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Delta target\nrange = {y_delta_arr.max()-y_delta_arr.min():.0f} kcal/mol')

plt.tight_layout()

In [ ]:
import collections.abc
collections.Iterable = collections.abc.Iterable
from dscribe.descriptors import SOAP as SOAP_QM7

# QM7 contains H, C, N, O, S — we need a descriptor that covers all species
soap_qm7_desc = SOAP_QM7(
    species=["H", "C", "N", "O", "S"],
    periodic=False,
    r_cut=5.0,
    n_max=4,
    l_max=4,
    sigma=0.3,
    average='inner',
)
print("Computing QM7 SOAP descriptors (this takes ~1–2 minutes)…")
soaps_qm7 = soap_qm7_desc.create(qm7)
print(f"Done. Descriptor shape: {soaps_qm7.shape}")

## 11. Homework Assignments

**Task for you**:
<div class="alert alert-block alert-info">

Use the code above and your newly acquired skills to build the best possible KRR model for the QM7 dataset to predict the PBE0 energy. The SOAP descriptor and initial comparison are already set up in Section 10 — build on them.

1. **Direct prediction**: Predict the PBE0 atomisation energy using a well-optimised KRR model. Use `GridSearchCV` to find the best `alpha` and `gamma`. Can you beat the PRL 2012 benchmark of ~10 kcal/mol? (Phys. Rev. Lett. 108, 058301)

2. **Delta learning**: Predict the PBE0–DFTB3 delta energy and add back the DFTB3 label. By what factor does delta learning reduce the error compared to direct learning (if at all)? Does hyperparameter optimisation help more for the direct or the delta model?

3. **Uncertainty**: Apply the ensemble method from Section 9 to your best model. Is the model more confident for small molecules (fewer heavy atoms) or large molecules? Plot the ensemble $\sigma$ as a function of the number of non-hydrogen atoms.

For all tasks:
- Use an 80%/20% train/test split with `random_state=42`
- Report the final MAE on the test set
</div>